In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from sklearn.model_selection  import StratifiedShuffleSplit

from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.compose import ColumnTransformer
from predict_function import predict

from sklearn.base import BaseEstimator, TransformerMixin


class Add_features(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.lon = 0
        self.lat = 1
        self.hma = 2
        self.trms = 3
        self.pop = 5
        self.med_inc = 7

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        lat_long = X[:, self.lon] + X[:, self.lat]
        hma_med_inc = X[:, self.hma] / X[:, self.med_inc]
        trms_pop = X[:, self.trms] / X[:, self.pop]

        return np.c_[X, lat_long, hma_med_inc, trms_pop]
        


house = pd.read_csv('./housing.csv')

X = house.drop('median_house_value', axis=1).copy()
y = house.median_house_value


split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=123)
(train_idx, test_idx), = split.split(X, X['ocean_proximity'])

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]


num_features = X.select_dtypes('float').columns
cat_features = X.select_dtypes('object').columns


num_pipline = Pipeline([
    ('impute', SimpleImputer()),
    ('add_feature', Add_features()),
    ('scale', StandardScaler())
])

cat_pipeline = Pipeline([
    ('one_hot_encoder', OneHotEncoder(sparse_output=False))
])


final_pipeline = ColumnTransformer([
    ('num_pipeline', num_pipline, num_features),
    ('cat_pipeline', cat_pipeline, cat_features)
])

final_pipeline.fit(X_train)

X_train_tr = final_pipeline.transform(X_train)
X_test_tr = final_pipeline.transform(X_test)



models = [
    ('Random_forest', RandomForestRegressor(n_jobs=-1))
]

predict(models, X_train_tr, X_test_tr, y_train, y_test)


Training model: Random_forest
Random_forest - R² Score: 0.8296
Random_forest - MAE: 29863.03

Summary:
Random_forest        | R²: 0.8296 | MAE: 29863.03


In [2]:
from sklearn.model_selection import KFold

In [3]:
folds = KFold()

In [4]:
folds.split(X_train_tr)

<generator object _BaseKFold.split at 0x0000018AD5997880>

In [5]:
c = 1
for i in folds.split(X_train_tr):
    print(f"Split no: {c}: {i}")
    print()
    c+=1


Split no: 1: (array([ 3303,  3304,  3305, ..., 16509, 16510, 16511]), array([   0,    1,    2, ..., 3300, 3301, 3302]))

Split no: 2: (array([    0,     1,     2, ..., 16509, 16510, 16511]), array([3303, 3304, 3305, ..., 6603, 6604, 6605]))

Split no: 3: (array([    0,     1,     2, ..., 16509, 16510, 16511]), array([6606, 6607, 6608, ..., 9905, 9906, 9907]))

Split no: 4: (array([    0,     1,     2, ..., 16509, 16510, 16511]), array([ 9908,  9909,  9910, ..., 13207, 13208, 13209]))

Split no: 5: (array([    0,     1,     2, ..., 13207, 13208, 13209]), array([13210, 13211, 13212, ..., 16509, 16510, 16511]))



In [6]:
X_train_tr

array([[ 0.62286403, -0.76422645,  1.70268675, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.56806326, -0.68937821,  0.4311512 , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.66271915, -0.75954843,  1.54374481, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.62286403, -0.77358248,  1.22586092, ...,  0.        ,
         0.        ,  0.        ],
       [ 1.21072692, -1.19928181, -1.31721019, ...,  0.        ,
         0.        ,  0.        ],
       [-0.85675696,  1.05552126, -0.20461658, ...,  0.        ,
         0.        ,  0.        ]])

In [7]:
y_train

5015     120900.0
3769     353600.0
4895     107500.0
7963     167100.0
19675    110700.0
           ...   
3500     179700.0
876      158000.0
5040     108600.0
15540    199600.0
16420    162500.0
Name: median_house_value, Length: 16512, dtype: float64

In [8]:
models = [
    ('Random_forest', RandomForestRegressor())
]

In [9]:
c = 1
for train_idx, test_idx in folds.split(X_train_tr):
    print(f"Split no: {c}")
    tr_X = X_train_tr[train_idx]
    ts_X = X_train_tr[test_idx]
    tr_y = y_train.values[train_idx]
    ts_y = y_train.values[test_idx]
    predict(models, tr_X, ts_X, tr_y, ts_y)
    c+=1


Split no: 1

Training model: Random_forest
Random_forest - R² Score: 0.8283
Random_forest - MAE: 31016.38

Summary:
Random_forest        | R²: 0.8283 | MAE: 31016.38
Split no: 2

Training model: Random_forest
Random_forest - R² Score: 0.8376
Random_forest - MAE: 30212.33

Summary:
Random_forest        | R²: 0.8376 | MAE: 30212.33
Split no: 3

Training model: Random_forest
Random_forest - R² Score: 0.8510
Random_forest - MAE: 29162.80

Summary:
Random_forest        | R²: 0.8510 | MAE: 29162.80
Split no: 4

Training model: Random_forest
Random_forest - R² Score: 0.8273
Random_forest - MAE: 30573.06

Summary:
Random_forest        | R²: 0.8273 | MAE: 30573.06
Split no: 5

Training model: Random_forest
Random_forest - R² Score: 0.8333
Random_forest - MAE: 29995.98

Summary:
Random_forest        | R²: 0.8333 | MAE: 29995.98


In [10]:
model = RandomForestRegressor(n_jobs=-1)

In [11]:
from sklearn.model_selection import cross_val_score

In [12]:
cross_val_r2_score = cross_val_score(model, X_train_tr, y_train, scoring='r2', cv=5)
cross_val_r2_score

array([0.82790881, 0.83572956, 0.85010417, 0.82725855, 0.83507118])

In [13]:
cross_val_nmse = cross_val_score(model, X_train_tr, y_train, scoring='neg_mean_squared_error', cv=5)
cross_val_nmse

array([-2.30934787e+09, -2.22350903e+09, -1.99280660e+09, -2.26598952e+09,
       -2.17836074e+09])

In [14]:
np.sqrt(-(cross_val_nmse))

array([48055.67468773, 47154.09875926, 44640.86242267, 47602.41085978,
       46672.91226449])

In [15]:
cross_val_nmae = cross_val_score(model, X_train_tr, y_train, scoring='neg_mean_absolute_error', cv=5)
cross_val_nmae

array([-30912.89372691, -30094.30043597, -29100.6291914 , -30485.62073895,
       -29703.96276802])

In [16]:
-(cross_val_nmae)

array([30912.89372691, 30094.30043597, 29100.6291914 , 30485.62073895,
       29703.96276802])

In [17]:
error = np.sqrt(-(cross_val_nmse))
error

array([48055.67468773, 47154.09875926, 44640.86242267, 47602.41085978,
       46672.91226449])

In [18]:
error.mean()

46825.19179878441

In [19]:
error.std()

1184.9743691162923